In [1]:
import csv
from PIL import ImageDraw, ImageFont
import re
from pathlib import Path
from dataset.augmentation import *

OUT_DIR = Path("dataset")
IMG_DIR = OUT_DIR / "images"
LABELS = OUT_DIR / "labels.csv"

WORDS = open("dataset/english_words.txt").read().splitlines() # 100.000 english words

random.seed(12345) # fixed seed for reproducibility

In [4]:
def random_word_or_string():
    if random.random() < 0.8:
        # Generates a random string of characters sampled from ALPHABET
        n = random.randint(MIN_LEN, MAX_LEN)
        return "".join(random.choice(ALPHABET_FOR_GENERATING) for _ in range(n))
    else:
        # Random real words
        words = [random.choice(WORDS) for _ in range(random.randint(1, 2))]
        if len("".join(words)) > 12:
            return words[0]
        return " ".join(words)

def clean_text(s: str) -> str:
    # Strips leading/trailing whitespace and collapses multiple spaces into one
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

def render_image(text: str, difficulty: str):
    """
    Renders a grayscale image containing the given text with random augmentations:
    """
    # Random light background (near-white grayscale)
    bg = random.randint(230, 255)
    img = Image.new("L", (IMG_W, IMG_H), bg)

    if random.random() < 0.05 and difficulty != "easy":
        img = gradient_bg()

    draw = ImageDraw.Draw(img)

    font_path = random.choice(FONTS)

    font_size = 32
    font = ImageFont.truetype(font_path, font_size)

    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    while text_width > IMG_W - 6 and font_size > 24:
        font_size -= 1
        try:
            font = ImageFont.truetype(font_path, font_size)
        except OSError:
            font = ImageFont.load_default()
        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]

    # Center text horizontally and vertically
    y_jitter = random.randint(-4, 4)
    x = (IMG_W - text_width) // 2 - bbox[0]
    y = (IMG_H - text_height) // 2 - bbox[1] + y_jitter

    LETTER_SPACING = random.randint(0, 3)

    if difficulty == "hard" and random.random() < 0.3:
        LETTER_SPACING = random.randint(-1,5)

    # Draw characters one by one to get character letter spacing
    x_cursor = x
    for ch in text:
        if random.random() < 0.3:
            for dx in (0, 1):
                for dy in (0, 1):
                    draw.text((x_cursor + dx, y + dy), ch, fill=0, font=font)
        else:
            draw.text((x_cursor, y), ch, fill=0, font=font)

        ch_bbox = draw.textbbox((0, 0), ch, font=font)
        ch_w = ch_bbox[2] - ch_bbox[0]
        x_cursor += ch_w + LETTER_SPACING


    if difficulty == "easy":
        if random.random() < 0.3:
            img = noise(img, 10, 30)

        if random.random() < 0.5:
            img = rotate_img(img, 1, bg)

        return img

    elif difficulty == "medium":
        augmentations = 0

        if random.random() < 0.3:
            img = noise(img, 50, 200)
            augmentations += 1

        if random.random() < 0.1:
            img = gaussian_blur(img, 0.25)
            augmentations += 1

        if random.random() < 0.2:
            img = contrast(img)
            augmentations += 1

        if random.random() < 0.2 and augmentations <= 2:
            img = gamma(img)
            augmentations += 1

        if random.random() < 0.2 and augmentations <= 2:
            img = brightness(img)
            augmentations += 1

        if random.random() < 0.5:
            img = rotate_img(img, 2, bg)

    elif difficulty == "hard":
        augmentations = 0

        if random.random() < 0.3:
            img = gaussian_blur(img, 0.5)
            augmentations += 1

        if random.random() < 0.3:
            img = contrast(img)
            augmentations += 1

        if random.random() < 0.3:
            img = gamma(img)
            augmentations += 1

        if random.random() < 0.2 and augmentations <= 2:
            img = brightness(img)
            augmentations += 1

        if random.random() < 0.3 and augmentations <= 3:
            img = perspective(img)
            augmentations += 1

        if random.random() < 0.5 and augmentations <= 4:
            img = noise(img, 250, 350)

        if random.random() < 0.2:
            img = rotate_img(img, 3, bg)

    return img

In [5]:
OUT_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

with open(LABELS, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "text"])

    easy = int(N_IMAGES * DIFFICULTY_RATIO[0])
    medium = int(N_IMAGES * DIFFICULTY_RATIO[1])
    hard = int(N_IMAGES * DIFFICULTY_RATIO[2])

    for i in range(N_IMAGES):
        if i < easy:
            difficulty = "easy"
        elif i < medium:
            difficulty = "medium"
        else:
            difficulty = "hard"

        if i % 10000 == 0:
            print(f"Generated {i} images")

        while True:
            text = clean_text(random_word_or_string())
            if text:
                break

        img = render_image(text, difficulty)

        fname = f"{i:06d}.png"
        img.save(IMG_DIR / fname, optimize=True)
        writer.writerow([fname, text])

print("DONE")
print("Images:", IMG_DIR)
print("Labels:", LABELS)


Generated 0 images
Generated 10000 images
Generated 20000 images
Generated 30000 images
Generated 40000 images
Generated 50000 images
Generated 60000 images
Generated 70000 images
Generated 80000 images
Generated 90000 images
DONE
Images: dataset\images
Labels: dataset\labels.csv
